In [1]:
import torch
from torch.utils.data import Dataset, ConcatDataset
import zarr
from torch.nn.functional import pad
import numpy as np
import os

In [ ]:


folder_path = '../../data/v1/2_processed/'
zarr_files = [
    os.path.join(folder_path, file)
    for file in os.listdir(folder_path)
    if file.endswith('.zarr')
]

print(zarr_files)

['../../data/v1/2_processed/s1a-iw-raw-s-vh-20250605t043047-20250605t043120-059506-07632c.zarr', '../../data/v1/2_processed/s1a-iw-raw-s-vh-20210314t085843-20210314t085915-036992-045a59.zarr', '../../data/v1/2_processed/s1b-iw-raw-s-vv-20210103t152235-20210103t152307-024992-02f972.zarr', '../../data/v1/2_processed/s1a-iw-raw-s-vh-20211012t204336-20211012t204408-040091-04bf46.zarr', '../../data/v1/2_processed/s1a-iw-raw-s-vh-20250615t012451-20250615t012523-059650-076814.zarr', '../../data/v1/2_processed/s1a-iw-raw-s-vv-20240508t030608-20240508t030640-053774-0688ca.zarr', '../../data/v1/2_processed/s1a-iw-raw-s-vh-20220702t112227-20220702t112259-043921-053e47.zarr', '../../data/v1/2_processed/s1b-iw-raw-s-vh-20190310t054645-20190310t054717-015288-01c9c5.zarr', '../../data/v1/2_processed/s1b-iw-raw-s-vh-20210728t234755-20210728t234827-028001-035728.zarr', '../../data/v1/2_processed/s1a-iw-raw-s-vh-20230213t104255-20230213t104327-047216-05aa65.zarr', '../../data/v1/2_processed/s1a-iw-raw-s

In [2]:
class ZarrDataset(Dataset):
    def __init__(self, zarr_path, target_shape_echo=(1600, 24100), target_shape_rfi=(10, 1600, 24100)):
        self.zarr = zarr.open(zarr_path, mode='r')
        self.echo = []
        self.rfi = []
        for key in self.zarr.keys():
            zarr_arrays = self.zarr[key] 
            echo = torch.from_numpy(zarr_arrays['echo'][:])
            rfi = torch.from_numpy(zarr_arrays['rfi'][:])
            padded_echo = self._pad_tensor(echo, target_shape_echo)
            padded_rfi = self._pad_tensor(rfi, target_shape_rfi)
            self.echo.append(padded_echo)
            self.rfi.append(padded_rfi)
        self.num_samples = len(self.echo)

    def _pad_tensor(self, tensor, target_shape):
            """
            Pad a tensor to the target shape.
            Args:
                tensor (torch.Tensor): Input tensor.
                target_shape (tuple): Target shape (e.g., (C, H, W)).
            Returns:
                torch.Tensor: Padded tensor.
            """
            # Calculate padding for each dimension
            pad_dims = []
            for i, (current_dim, target_dim) in enumerate(zip(tensor.shape, target_shape)):
                pad_size = target_dim - current_dim
                if pad_size < 0:
                    raise ValueError(f"Target dimension {target_dim} is smaller than current dimension {current_dim}.")
                pad_dims.extend([0, pad_size])  # Pad at the end of each dimension

            # Pad the tensor
            padded_tensor = pad(tensor, pad_dims, mode='constant', value=0)
            return padded_tensor

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.echo[idx], self.rfi[idx]

In [2]:

class SARZarrDataset(Dataset):
    def __init__(self, root_dir, array_type='echo'):
        """
        Args:
            root_dir (str): Path to the directory containing all .zarr files.
            array_type (str): Type of array to load ('echo' or 'RFI').
        """
        self.root_dir = root_dir
        self.array_type = array_type
        self.zarr_files = []
        self.bursts_per_file = []
        self.cumulative_bursts = [0]

        # Scan all .zarr files and count bursts
        for file_name in sorted(os.listdir(root_dir)):
            if file_name.endswith('.zarr'):
                file_path = os.path.join(root_dir, file_name)
                z = zarr.open(file_path, mode='r')
                bursts = [key for key in z.keys() if key.startswith('burst_')]
                self.zarr_files.append(z)
                self.bursts_per_file.append(len(bursts))
                self.cumulative_bursts.append(self.cumulative_bursts[-1] + len(bursts))

        self.total_bursts = self.cumulative_bursts[-1]

    def __len__(self):
        return self.total_bursts

    def __getitem__(self, idx):
        # Find which file and burst the index corresponds to
        file_idx = 0
        while idx >= self.cumulative_bursts[file_idx + 1]:
            file_idx += 1
        burst_idx = idx - self.cumulative_bursts[file_idx]

        # Load the burst
        z = self.zarr_files[file_idx]
        burst_key = f'burst_{burst_idx}'
        array = z[burst_key][self.array_type]

        # Convert to tensor
        tensor = torch.from_numpy(array[:])
        return tensor

    def close(self):
        """Close all open Zarr files."""
        for z in self.zarr_files:
            z.store.close()

In [3]:
dataset = SARZarrDataset(root_dir='../../data/v1/2_processed/', array_type='echo')

KeyboardInterrupt: 

In [3]:
zarr_group = zarr.open('../../data/v1/2_processed/s1a-iw-raw-s-hv-20220912t101317-20220912t101349-044970-055f55.zarr', mode='r')


In [4]:
#zarr_array =
#zarr_group = zarr.open('../../data/v1/2_processed/s1a-iw-raw-s-hv-20220912t101317-20220912t101349-044970-055f55.zarr', mode='r')
'''
for key in zarr_group.keys():
    zarr_array = zarr_group[key] 
    array_names = list(zarr_array.arrays())
    print(f"Arrays in {key}: {array_names}")
    print(torch.from_numpy(zarr_array['echo'][:]).shape)
    print(torch.from_numpy(zarr_array['rfi'][:]).shape)
    #print(type(zarr_array))
    #print(zarr_array.tree())
    for i in range(len(zarr_array)):
        numpy_array = zarr_array[i]  # Get the i-th sample as NumPy
        tensor = torch.from_numpy(numpy_array)  # Convert to tensor
        print(f"Sample {i}: Tensor shape {tensor.shape}")
    
    for array in zarr_array.arrays():
        #print(type(array[1][0]))
        tensor = torch.from_numpy(array[1][:])
        print(tensor.shape)
'''
    #zarr_array += z[key]

'\nfor key in zarr_group.keys():\n    zarr_array = zarr_group[key] \n    array_names = list(zarr_array.arrays())\n    print(f"Arrays in {key}: {array_names}")\n    print(torch.from_numpy(zarr_array[\'echo\'][:]).shape)\n    print(torch.from_numpy(zarr_array[\'rfi\'][:]).shape)\n    #print(type(zarr_array))\n    #print(zarr_array.tree())\n    for i in range(len(zarr_array)):\n        numpy_array = zarr_array[i]  # Get the i-th sample as NumPy\n        tensor = torch.from_numpy(numpy_array)  # Convert to tensor\n        print(f"Sample {i}: Tensor shape {tensor.shape}")\n\n    for array in zarr_array.arrays():\n        #print(type(array[1][0]))\n        tensor = torch.from_numpy(array[1][:])\n        print(tensor.shape)\n'

In [5]:
dataset1 = ZarrDataset('../../data/v1/2_processed/s1a-iw-raw-s-hv-20220912t101317-20220912t101349-044970-055f55.zarr')
#dataset2 = ZarrDataset('../../data/v1/2_processed/data2.zarr')
#dataset3 = ZarrDataset('../../data/v1/2_processed/data3.zarr')

RuntimeError: [enforce fail at alloc_cpu.cpp:124] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 93837004800 bytes. Error code 12 (Cannot allocate memory)

In [ ]:
for data in dataset1:
    print(data.shape)
    break